In [ ]:
pip install tensorflow

In [ ]:
import tensorflow as tf
print(tf.__version__)  # 예: 2.15.0

from tensorflow import keras
from tensorflow.keras import layers

In [ ]:
# 3-1. Sequential API

from tensorflow.keras import Sequential
from tensorflow.keras.layers import Dense, Dropout

model = Sequential([
    Dense(128, activation="relu", input_shape=(4,)),  # 입력층 + 첫 번째 은닉층
    Dense(64, activation="relu"),                      # 두 번째 은닉층
    Dense(3, activation="softmax")                     # 출력층 (클래스 3개)
])


In [ ]:
# 3-3. 활성화 함수

# ReLU — 은닉층에서 가장 많이 쓰임
Dense(128, activation="relu")

# Sigmoid — 이진 분류 출력층 (0~1 사이 확률)
Dense(1, activation="sigmoid")

# Softmax — 다중 분류 출력층 (확률의 합 = 1)
Dense(10, activation="softmax")  # 10개 클래스

In [ ]:
# 3-4. 모델 구조 확인

model.summary()

In [ ]:
# 4-1. 컴파일 설정

model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

In [ ]:
# 4-4. 모델 학습

history = model.fit(
    X_train, y_train,
    epochs=50,
    batch_size=32,
    validation_split=0.2,
    verbose=1
)

In [ ]:
# 4-5. 학습 곡선 시각화

import matplotlib.pyplot as plt

plt.plot(history.history["accuracy"],     label="훈련 정확도")
plt.plot(history.history["val_accuracy"], label="검증 정확도", linestyle="--")
plt.title("정확도 변화")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.legend()
plt.grid(True)
plt.show()

plt.plot(history.history["loss"],     label="훈련 손실")
plt.plot(history.history["val_loss"], label="검증 손실", linestyle="--")
plt.title("손실 변화")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
# 5-2. MNIST  데이터 전처리

from tensorflow.keras.datasets import mnist

(X_train, y_train), (X_test, y_test) = mnist.load_data()

print(X_train.shape)  # (60000, 28, 28)
print(y_train.shape)  # (60000,)

import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 5, figsize=(12, 5))
for i, ax in enumerate(axes.flat):
    ax.imshow(X_train[i], cmap="gray")
    ax.set_title(f"정답: {y_train[i]}")
    ax.axis("off")
plt.show()

# 정규화
X_train = X_train / 255.0
X_test  = X_test  / 255.0

# Flatten
X_train = X_train.reshape(-1, 784)
X_test  = X_test.reshape(-1, 784)

print(X_train.shape)  # (60000, 784)

In [ ]:
# 5-3. 모델 설계 및 학습

from tensorflow.keras import Sequential
from tensorflow.keras.layers import Dense, Dropout

model = Sequential([
    Dense(256, activation="relu", input_shape=(784,)),
    Dropout(0.3),
    Dense(128, activation="relu"),
    Dropout(0.3),
    Dense(10, activation="softmax")
])

model.summary()

model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

history = model.fit(
    X_train, y_train,
    epochs=20,
    batch_size=64,
    validation_split=0.2
)

In [ ]:
# 5-4. 예측 결과 시각화
import numpy as np

y_pred_proba = model.predict(X_test)
y_pred       = np.argmax(y_pred_proba, axis=1)

fig, axes = plt.subplots(2, 5, figsize=(14, 6))
for i, ax in enumerate(axes.flat):
    ax.imshow(X_test[i].reshape(28, 28), cmap="gray")
    color = "blue" if y_pred[i] == y_test[i] else "red"
    ax.set_title(f"예측:{y_pred[i]} / 정답:{y_test[i]}", color=color)
    ax.axis("off")

plt.suptitle("파란색: 정답, 빨간색: 오답")
plt.show()

In [ ]:
# 5-5. 정확도 확인
test_loss, test_acc = model.evaluate(X_test, y_test)
print(f"테스트 정확도: {test_acc:.4f}")  # 예: 0.9789 (97.89%)

In [ ]:
# 6-2. Dropout

from tensorflow.keras.layers import Dropout

Dropout(0.3)  # 30%의 뉴런을 랜덤으로 비활성화
# 예측할 때는 모든 뉴런이 다시 켜집니다

In [ ]:
# 6-3. EarlyStopping

from tensorflow.keras.callbacks import EarlyStopping

early_stop = EarlyStopping(
    monitor="val_loss",
    patience=5,
    restore_best_weights=True
)

history = model.fit(
    X_train, y_train,
    epochs=100,
    validation_split=0.2,
    callbacks=[early_stop]
)

print(f"실제 학습된 에폭 수: {len(history.history['loss'])}")

In [ ]:
# 7-1. 모델 저장

model.save("mnist_model.keras")

In [ ]:
# 7-2. 모델 불러오기

from tensorflow.keras.models import load_model

loaded_model = load_model("mnist_model.keras")

y_pred = loaded_model.predict(X_test)
print(f"정확도: {loaded_model.evaluate(X_test, y_test)[1]:.4f}")